# SOEN Disruption Prediction — PCA-8 seq2seq Binary Classification

Train a neuromorphic SOEN model on the PCA-8 + 100× decimated + flattop dataset for **per-timestep binary disruption classification**, directly comparable to the TCN baseline.

**Task**: `seq2seq` with `cross_entropy` loss — predict `{0=clear, 1=disruptive}` at every timestep.

**Data**: Same `all_data.h5` from `preprocessing_pca8_100x_flattop.ipynb` → `(N, 8, 7812)`.

**Model**: SOEN `8IN→28H(rec)→2ClassReadout` with hardware constraints:
- On-chip connections (J_0_to_1, J_1_to_1): trainable, 3-bit QAT, bounds [-0.14, 0.14]
- Physics params (phi_offset, bias_current, gamma): **fixed** per hardware spec
- Hidden→readout (J_1_to_2): **fixed** one-to-one coupling J=0.5

**Deliverable**: Quantized `.soen` checkpoint for hardware deployment.

In [ ]:
import sys
import shutil
from pathlib import Path

import numpy as np
import h5py
import yaml

# ── Locate soen-toolkit ──────────────────────────────────────────
# Try multiple candidate locations (local dev, SciServer, etc.)
_CANDIDATES = [
    Path("/Users/davidpark/Documents/Cursor/soenhardware/soen-toolkit/src"),
    Path("/home/idies/workspace/Temporary/dpark1/scratch/soenhardware/soen-toolkit/src"),
    Path("/home/idies/workspace/Temporary/dpark1/scratch/soen-toolkit/src"),
    Path.home() / "soen-toolkit/src",
]

SOEN_TOOLKIT_SRC = None
for _c in _CANDIDATES:
    if (_c / "soen_toolkit").is_dir():
        SOEN_TOOLKIT_SRC = _c
        break

if SOEN_TOOLKIT_SRC is None:
    raise FileNotFoundError(
        "soen-toolkit not found. Tried:\n" +
        "\n".join(f"  {c}" for c in _CANDIDATES) +
        "\nClone it or adjust _CANDIDATES above."
    )

TUTORIAL_DIR = SOEN_TOOLKIT_SRC / "soen_toolkit/tutorial_notebooks/time_to_event_tutorial"

# Add to sys.path
for p in [str(SOEN_TOOLKIT_SRC), str(TUTORIAL_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Also copy notebook_utils.py next to this notebook as a fallback
_local_utils = Path("notebook_utils.py")
_src_utils = TUTORIAL_DIR / "notebook_utils.py"
if _src_utils.exists() and (not _local_utils.exists() or _local_utils.stat().st_mtime < _src_utils.stat().st_mtime):
    shutil.copy2(_src_utils, _local_utils)
    print(f"Copied notebook_utils.py from {_src_utils}")

import notebook_utils
print(f"notebook_utils loaded from: {notebook_utils.__file__}")

# ── Paths ─────────────────────────────────────────────────────────
PCA8_H5 = Path("/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/pca8_100x_flattop/all_data.h5")

SOEN_DIR = Path("soen_training")
DATASET_DIR = SOEN_DIR / "datasets"
MODEL_DIR = SOEN_DIR / "model_specs"
CONFIG_DIR = SOEN_DIR / "training_configs"
RESULTS_DIR = SOEN_DIR / "results"

for d in [DATASET_DIR, MODEL_DIR, CONFIG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"soen-toolkit: {SOEN_TOOLKIT_SRC}")
print(f"PCA8 source:  {PCA8_H5}")
print(f"SOEN dir:     {SOEN_DIR.resolve()}")

## 1. Convert PCA8 H5 → SOEN HDF5 format

SOEN expects `(N, T, D)` layout with `labels` as int64 for classification.

| PCA8 H5 | SOEN H5 |
|---------|---------|
| `X` (N, 8, 7812) float32 | `data` (N, 7812, 8) float32 |
| `target` (N, 7812) float32 {0,1} | `labels` (N, 7812) int64 {0,1} |
| `weight` (N, 7812) float32 {0,1} | `target_mask` (N, 7812) bool |
| — | `input_mask` (N, 7812) bool (all True) |

In [ ]:
SOEN_H5 = DATASET_DIR / "pca8_disruption_seq2seq_2class.h5"

with h5py.File(PCA8_H5, "r") as src, h5py.File(SOEN_H5, "w") as dst:
    for split in ("train", "val", "test"):
        g = dst.create_group(split)

        # X: (N, 8, T) → (N, T, 8) — SOEN expects channels-last
        X = np.asarray(src[f"{split}/X"])          # (N, 8, 7812)
        X = np.transpose(X, (0, 2, 1))             # (N, 7812, 8)
        g.create_dataset("data", data=X, dtype=np.float32)

        # target: float32 {0.0, 1.0} → int64 {0, 1}
        target = np.asarray(src[f"{split}/target"])  # (N, 7812)
        labels = target.astype(np.int64)
        g.create_dataset("labels", data=labels, dtype=np.int64)

        # weight → target_mask: where loss should be computed
        weight = np.asarray(src[f"{split}/weight"])  # (N, 7812)
        target_mask = (weight > 0).astype(bool)
        g.create_dataset("target_mask", data=target_mask)

        # input_mask: all timesteps are valid (pre-subsequenced)
        input_mask = np.ones(X.shape[:2], dtype=bool)
        g.create_dataset("input_mask", data=input_mask)

        N, T, D = X.shape
        n_pos = int(labels.sum())
        n_valid = int(target_mask.sum())
        print(f"  {split}: N={N}, T={T}, D={D}, "
              f"pos_labels={n_pos}/{n_valid} valid ({n_pos/max(n_valid,1)*100:.1f}%)")

print(f"\nSaved: {SOEN_H5}")

## 2. Build SOEN model

Architecture: `Linear(8) → SingleDendrite(28, recurrent) → DendriteReadout(2)`

**Fixed** (hardware spec): phi_offset=0.23, bias_current=1.7, gamma_plus/minus, J_1_to_2=0.5
**Trainable** (3-bit QAT): J_0_to_1 (input→hidden), J_1_to_1 (recurrent)

In [ ]:
from notebook_utils import build_direct_readout_model

MODEL_PATH = MODEL_DIR / "8IN_28H_2ClassSeq2Seq.soen"

build_direct_readout_model(
    model_path=MODEL_PATH,
    output_dim=2,          # binary: {0=clear, 1=disruptive}
    input_dim=8,           # 8 PCA components
    hidden_dim=28,         # standard SOEN hidden size
    dt_ns=10.0,            # hardware timestep
    # ── Fixed hardware parameters ──
    phi_offset=0.23,
    phi_offset_learnable=False,
    bias_current=1.7,
    bias_current_learnable=False,
    gamma_plus=2.3508e-5,
    gamma_minus=2.6995e-5,
    gamma_minus_learnable=False,
)

print(f"Model saved: {MODEL_PATH}")
print(f"  Input:  8 (PCA components)")
print(f"  Hidden: 28 (SingleDendrite, recurrent)")
print(f"  Output: 2 (binary per-timestep classification)")

## 3. Build training config

Write YAML matching the SOEN tutorial convention, but with:
- `mapping: seq2seq` + `losses: cross_entropy` → per-timestep binary classification
- `num_classes: 2`
- No `time_pooling` (seq2seq outputs at every timestep)
- QAT on J_0_to_1 and J_1_to_1 only (3-bit, [-0.14, 0.14])
- Fixed-length data (all subsequences are 7812)

In [ ]:
EXPERIMENT_NAME = "pca8_disruption_seq2seq_2class"
SEQ_LEN = 7812
DT_NS = 10.0
MAX_EPOCHS = 200
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
BACKEND = "jax"  # or "torch"

CONFIG_PATH = CONFIG_DIR / f"training_config_{EXPERIMENT_NAME}.yaml"

cfg = {
    "seed": 42,
    "run": {
        "mode": "train",
        "eval_split": "val",
        "eval_checkpoint_path": None,
    },
    "training": {
        "paradigm": "supervised",
        "mapping": "seq2seq",                     # per-timestep prediction
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "optimizer": {
            "name": "adam",
            "lr": LEARNING_RATE,
            "kwargs": {"weight_decay": 1e-4},
        },
        "gradient_clip_val": 1.0,
        "accelerator": "cpu",
        "precision": "32-true",
        "devices": "auto",
        "num_workers": 0,
        "num_repeats": 1,
        "checkpoint_every_n_epochs": 10,
        "checkpoint_save_top_k": 1,
        "checkpoint_save_last": True,
        "save_initial_state": True,               # saves initial.soen
        "save_soen_core_in_checkpoint": True,      # saves last.soen
        "accumulate_grad_batches": 1,
        "gradient_checkpointing": False,
        "ar": {"enabled": False},
        "losses": [{"name": "cross_entropy", "weight": 1.0}],  # per-timestep CE
        "run_test_after_fit": False,
    },
    "data": {
        "data_path": str(SOEN_H5.resolve()),
        "cache": {"strategy": "full"},
        "resample_mode": "interpolate",
        "input_encoding": "raw",
        "variable_length": False,                  # fixed-length 7812
        "target_seq_len": SEQ_LEN,
        "sequence_length": SEQ_LEN,
        "total_time_ns": float(SEQ_LEN) * DT_NS,  # 78120.0
        "num_classes": 2,                          # binary classification
        "pad_value": 0.0,
        "min_scale": 0.0,                          # min-max input scaling
        "max_scale": 1.0,
    },
    "model": {
        "base_model_path": str(MODEL_PATH.resolve()),
        "load_exact_model_state": False,
        "backend": BACKEND,
        # NO time_pooling for seq2seq — output at every timestep
    },
    "logging": {
        "project_dir": str(RESULTS_DIR.resolve()),
        "project_name": "bnl_training",
        "group_name": "baseline",
        "experiment_name": EXPERIMENT_NAME,
        "metrics": [],
        "log_freq": 10,
        "log_level": "INFO",
        "log_gradients": False,
        "track_connections": False,
        "track_layer_params": False,
        "log_batch_metrics": True,
        "track_power_metrics": False,
        "mlflow_active": False,
    },
    "callbacks": {
        "lr_scheduler": {
            "type": "linear",
            "max_lr": LEARNING_RATE,
            "min_lr": 1e-6,
            "log_space": True,
        },
        "qat": {
            "active": True,
            "min_val": -0.14,
            "max_val": 0.14,
            "bits": 3,                             # 3-bit quantization
            "connections": ["J_0_to_1", "J_1_to_1"],  # on-chip only
            "update_on_train_epoch_start": True,
            "stochastic_rounding": False,
        },
        "stateful_training": {
            "enable_for_training": True,
            "enable_for_validation": False,        # JAX limitation
            "sample_selection": "random",
            "per_sample": False,
            "verbose": False,
        },
    },
    "cloud": {"active": False},
}

with CONFIG_PATH.open("w") as f:
    yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"Config saved: {CONFIG_PATH}")
print(f"  mapping: seq2seq + cross_entropy (per-timestep binary)")
print(f"  seq_len: {SEQ_LEN}, dt_ns: {DT_NS}")
print(f"  QAT: 3-bit on J_0_to_1, J_1_to_1 (bounds [-0.14, 0.14])")
print(f"  epochs: {MAX_EPOCHS}, batch: {BATCH_SIZE}, lr: {LEARNING_RATE}")

## 4. Train

Invokes `soen_toolkit.training` via subprocess (same as tutorial `02_train_models.ipynb`).
Saves `initial.soen` and `last.soen` checkpoints in `.soen` format.

In [ ]:
from notebook_utils import run_training

print(f"Launching SOEN training: {CONFIG_PATH}")
print(f"  This may take a while for {MAX_EPOCHS} epochs on seq_len={SEQ_LEN}...")
run_training(CONFIG_PATH)

## 5. Plot loss curves

In [ ]:
import matplotlib.pyplot as plt
from notebook_utils import read_training_scalars_with_fallback

scalars = read_training_scalars_with_fallback(CONFIG_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
for tag in scalars:
    if "loss" in tag.lower() and "epoch" in tag.lower():
        df = scalars[tag]
        label = "train" if "train" in tag.lower() else "val"
        ax.plot(df["step"], df["value"], label=label)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy (if logged)
ax = axes[1]
found_acc = False
for tag in scalars:
    if "acc" in tag.lower() and "epoch" in tag.lower():
        df = scalars[tag]
        label = "train" if "train" in tag.lower() else "val"
        ax.plot(df["step"], df["value"], label=label)
        found_acc = True
if found_acc:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Per-timestep Accuracy")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No accuracy metrics logged", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Accuracy (not available)")

plt.tight_layout()
plt.show()

## 6. Audit trained weights and quantize to 3-bit `.soen`

Verify:
1. All trainable connections (J_0_to_1, J_1_to_1) are within [-0.14, 0.14]
2. Fixed parameters (phi_offset, bias_current, gamma, J_1_to_2) are unchanged
3. QAT produced valid 3-bit (9-level) quantized weights

**Output**: `last_quant_3bit9lvl.soen` — the hardware deliverable.

In [ ]:
from notebook_utils import audit_and_quantize_latest_run

audit = audit_and_quantize_latest_run(
    results_dir=RESULTS_DIR,
    config_path=CONFIG_PATH,
    experiment_name=EXPERIMENT_NAME,
    target_connections=["J_0_to_1", "J_1_to_1"],
    weight_min=-0.14,
    weight_max=0.14,
    quant_levels=9,  # 3-bit → 2^3+1 = 9 levels
)

print("=== Audit Results ===")
print(f"  Bounds OK:              {audit['bounds_ok']}")
print(f"  Fixed params unchanged: {audit['fixed_params_unchanged']}")
print(f"  QAT active in config:   {audit['qat_active_in_config']}")

print("\n=== Weight Bounds ===")
for conn, stats in audit["bounds_stats"].items():
    print(f"  {conn}: min={stats['min']:.6f}, max={stats['max']:.6f}, in_bounds={stats['in_bounds']}")

print("\n=== Quantization Levels ===")
for conn, n_lvl in audit.get("quant_levels_present", {}).items():
    print(f"  {conn}: {n_lvl} unique quantized values")

print("\n=== Loss (float vs quantized) ===")
loss_info = audit.get("loss_info", {})
print(f"  Train loss (float32):   {loss_info.get('train_loss_float', 'N/A')}")
print(f"  Train loss (quantized): {loss_info.get('train_loss_quantized', 'N/A')}")

print(f"\n=== Checkpoint ===")
print(f"  Dir:       {audit['checkpoint_dir']}")
print(f"  Quantized: {audit['quantized_checkpoint']}")
print(f"\n  Hardware deliverable: {audit['quantized_checkpoint']}")

## 7. Verify checkpoint structure

Load the quantized `.soen` file and inspect its contents to confirm it matches the expected format for hardware deployment.

In [ ]:
import torch

quant_path = Path(audit["quantized_checkpoint"])
obj = torch.load(quant_path, map_location="cpu", weights_only=False)

print(f"=== .soen checkpoint: {quant_path.name} ===")
print(f"Top-level keys: {list(obj.keys())}")
print(f"\nmodel_type: {obj.get('model_type', 'N/A')}")
print(f"dt_ns:      {obj.get('dt_ns', 'N/A')}")

print(f"\n=== state_dict keys ===")
sd = obj.get("state_dict", {})
for k, v in sd.items():
    if hasattr(v, "shape"):
        print(f"  {k}: shape={tuple(v.shape)}, dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v).__name__} = {v}")

print(f"\n=== Layers ===")
for layer_cfg in obj.get("layers_config", []):
    print(f"  Layer {layer_cfg.get('id', '?')}: {layer_cfg.get('type', '?')} "
          f"dim={layer_cfg.get('dim', '?')} — {layer_cfg.get('description', '')}")

print(f"\n=== Connections ===")
for conn_cfg in obj.get("connections_config", []):
    src, tgt = conn_cfg.get("source_layer_id", "?"), conn_cfg.get("target_layer_id", "?")
    learnable = conn_cfg.get("learnable", "?")
    print(f"  {src}→{tgt}: structure={conn_cfg.get('structure',{}).get('type','?')}, "
          f"learnable={learnable}")